In [1]:
!pip install pycountry

In [2]:
import wbdata
import pandas as pd
import datetime
import os
import pycountry

Key '7570363343715761596' not in persistent cache.
Key '-7741520987170889531' not in persistent cache.
Key '6470197185421091227' not in persistent cache.
Key '7178263312451689529' not in persistent cache.
Key '3676632835777186222' not in persistent cache.
Key '5098189508331251761' not in persistent cache.
Key '5715349704679321370' not in persistent cache.
Key '132970029649205309' not in persistent cache.
Key '-3778199516162751027' not in persistent cache.
Key '3431161254338903068' not in persistent cache.
Key '-259130001614866758' not in persistent cache.
Key '7639954306785324726' not in persistent cache.
Key '9061510979339390265' not in persistent cache.
Key '-1722043609387234917' not in persistent cache.
Key '-2583291025715655081' not in persistent cache.
Key '-3846788665877122720' not in persistent cache.
Key '-3683457184010325450' not in persistent cache.
Key '-7373748439964095276' not in persistent cache.
Key '-6477021533654230455' not in persistent cache.
Key '-505546486110016491

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")

In [4]:
# 1. Define the indicators you want to fetch
# indicators = {
#     "SP.POP.GROW": "pop_growth",
#     "SP.POP.TOTL": "pop_total",
#     "SP.URB.TOTL.IN.ZS": "pop_urban_pct",
#     "SI.POV.DDAY": "poverty_headcount_ratio",
#     "EG.ELC.ACCS.ZS": "access_to_electricity_pct",
#     "EG.USE.ELEC.KH.PC": "electricity_consumption_per_capita_kwh",
#     "EG.USE.PCAP.KG.OE": "energy_use_per_capita_kg_of_oil_equivalent",
#     "EG.USE.COMM.GD.PP.KD": "energy_use_kg_of_oil_equivalent_per_gdp",
#     "EG.ELC.COAL.ZS": "electricity_from_coal_pct",
#     "EG.ELC.NGAS.ZS": "electricity_from_natural_gas_pct",
#     "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
#     "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
#     "AG.LND.FRST.ZS": "forest_area_pct",
#     "NV.AGR.TOTL.ZS": "agriculture_value_added_pct",
#     "ER.H2O.FWTL.ZS": "annual_freshwater_withdrawals_pct"
# }


indicators = {
    "SP.POP.TOTL": "pop_total",
    "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
    "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
    "AG.LND.FRST.ZS": "forest_area_pct",
    "NY.GDP.MKTP.KD": "gdp_2015_usd",
    "NE.IMP.GNFS.KD": "imports_2015_usd",
    "NV.IND.TOTL.KD": "industry_value_added_2015_usd",
    "NV.IND.MANF.KD": "manufacturing_value_added_2015_usd",
    "NE.EXP.GNFS.KD": "exports_2015_usd",
}

In [5]:
date_range = (datetime.datetime(2000,1,1), datetime.datetime(2022,1,1))

df = wbdata.get_dataframe(
    indicators,
    country="all",
    date=date_range,
    freq='Y',
    parse_dates=True
).reset_index().rename(columns={'country':'region','date':'year'})

# 2) Pull the country lookup (wbdata.get_countries returns a list of dicts)
country_list = wbdata.get_countries()  

#  Inspect one entry to see its keys:
# print(country_list[0])
# -> {'id': 'ABW', 'iso2Code': 'AW', 'name': 'Aruba', …}

# 3) Build a small DataFrame of name → ISO-2
country_df = pd.DataFrame(country_list)[['name','iso2Code']]
country_df.columns = ['region','iso2']  

# 4) Merge it onto your indicators DataFrame
df = df.merge(country_df, on='region', how='left')

# 5) (Optional) Convert ISO-2 → ISO-3 with pycountry
def iso2_to_iso3(c2):
    try:
        return pycountry.countries.get(alpha_2=c2).alpha_3
    except:
        return None

df['iso_alpha_3'] = df['iso2'].map(iso2_to_iso3)

# make year only the year
df['year'] = df['year'].dt.year

# drop iso2 and region
df = df.drop(columns=['iso2', 'region'])


Key '2628563875942455163' not in persistent cache.


In [6]:
df

,year,pop_total,electricity_from_renewables_pct,renewable_energy_consumption_pct,forest_area_pct,gdp_2015_usd,imports_2015_usd,industry_value_added_2015_usd,manufacturing_value_added_2015_usd,exports_2015_usd,iso_alpha_3
0,2022,731821393.0,NaN,NaN,29.737205,1.040350e+12,3.196559e+11,2.535995e+11,1.016612e+11,2.360208e+11,None
1,2021,713090928.0,4.939318,NaN,29.955194,1.004646e+12,2.769094e+11,2.468588e+11,9.989399e+10,2.143081e+11,None
2,2020,694446100.0,4.585792,66.123449,30.174252,9.606813e+11,2.448648e+11,2.363776e+11,9.446765e+10,1.958722e+11,None
3,2019,675950189.0,5.173796,63.387090,30.391626,9.890095e+11,2.770649e+11,2.482358e+11,1.014837e+11,2.217922e+11,None
4,2018,657801085.0,3.945917,62.242631,30.611512,9.677734e+11,2.838657e+11,2.437429e+11,1.002104e+11,2.275611e+11,None
...,...,...,...,...,...,...,...,...,...,...,...
6113,2004,12365896.0,0.893713,81.800000,46.999354,1.433634e+10,3.279565e+09,2.952811e+09,1.423221e+09,3.288381e+09,ZWE
6114,2003,12232323.0,0.595068,78.000000,47.118444,1.522027e+10,3.535782e+09,2.922498e+09,1.581357e+09,3.357422e+09,ZWE
6115,2002,12087653.0,0.815768,74.700000,47.237534,1.833658e+10,4.330718e+09,3.505607e+09,1.817651e+09,4.219842e+09,ZWE
6116,2001,11971901.0,1.686061,72.300000,47.356624,2.012665e+10,4.391100e+09,3.891460e+09,2.089255e+09,5.081993e+09,ZWE


In [7]:
df.to_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_indicators.csv'), index=False)